# WSI Benchmark EDA Across Cohorts

Explore clinical, fusion, and WSI ROC/PR AUC metrics across run-level groupings and compare whether tool types behave similarly in the vulvar and LUSC cohorts.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    sns = None

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 120)

if sns is not None:
    sns.set_theme(style="whitegrid", context="notebook")
else:
    plt.style.use("seaborn-v0_8-whitegrid")

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "outputs").exists():
            return p
    raise FileNotFoundError("Could not find wsi_recurrence repo root from current working directory.")

repo_root = find_repo_root()
data_paths = [
    repo_root / "outputs" / "benchmarks" / "vulvar_all_results_wide.csv",
    repo_root / "outputs" / "benchmarks" / "lusc_all_results_wide.csv",
]
data_paths

In [ ]:
frames = []
for path in data_paths:
    cohort_df = pd.read_csv(path)
    if "cohort" not in cohort_df.columns or cohort_df["cohort"].isna().all():
        cohort_df["cohort"] = path.name.split("_all_results_wide.csv")[0]
    cohort_df["source_csv"] = str(path.relative_to(repo_root))
    frames.append(cohort_df)

df = pd.concat(frames, ignore_index=True)

metric_cols = [
    "clinical_roc_auc",
    "fusion_roc_auc",
    "wsi_roc_auc",
    "clinical_pr_auc",
    "fusion_pr_auc",
    "wsi_pr_auc",
]

group_cols = ["mode", "tool_type", "tile_aggregator", "slide_encoder", "experiment_base"]
required = [*group_cols, *metric_cols, "model", "run_name", "run_dir"]
base_required = ["cohort", "mode", "tile_aggregator", "slide_encoder", "experiment_base", *metric_cols, "model", "run_name", "run_dir"]
missing = sorted(set(base_required) - set(df.columns))
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

for col in ["cohort", "mode", "tile_aggregator", "slide_encoder", "experiment_base"]:
    df[col] = df[col].fillna("not_applicable").astype(str)

df["tool_type"] = np.where(
    df["mode"].eq("tile"),
    "tile:" + df["tile_aggregator"],
    np.where(df["mode"].eq("slide"), "slide:" + df["slide_encoder"], df["mode"]),
)
df["tool_family"] = df["tool_type"].str.split(":", n=1).str[0]

print(f"Loaded {len(df):,} rows from {len(data_paths)} benchmark files")
display(df.groupby(["cohort", "source_csv"]).size().rename("rows").reset_index())
display(df.head())
display(df[["cohort", *group_cols, *metric_cols]].describe(include="all"))

## Case Counts

Summarize the number of unique cases used by each benchmark result set, split by recurrence status. WSI counts come from `all_predictions_*.csv`; clinical/fusion counts come from `fusion_predictions.csv` when available.

In [ ]:
LABEL_CANDIDATES = ["has_recurrence", "recur", "recurrence (1=yes)", "label", "target"]
ID_CANDIDATES = ["patient", "patient_id", "pred_id", "slide_id", "case_id"]


def first_existing(columns: pd.Index, candidates: list[str]) -> str | None:
    return next((c for c in candidates if c in columns), None)


def case_counts_from_predictions(path: Path) -> dict:
    pred_df = pd.read_csv(path)
    label_col = first_existing(pred_df.columns, LABEL_CANDIDATES)
    id_col = first_existing(pred_df.columns, ID_CANDIDATES)
    if label_col is None:
        raise ValueError(f"Could not find a recurrence label column in {path}")
    if id_col is None:
        raise ValueError(f"Could not find a case identifier column in {path}")

    cases = pred_df[[id_col, label_col]].dropna(subset=[id_col, label_col]).copy()
    cases[label_col] = pd.to_numeric(cases[label_col], errors="coerce")
    cases = cases.dropna(subset=[label_col]).drop_duplicates(subset=[id_col])
    cases[label_col] = cases[label_col].astype(int)

    n_recurrence = int((cases[label_col] == 1).sum())
    n_no_recurrence = int((cases[label_col] == 0).sum())
    return {
        "n_total": int(len(cases)),
        "n_recurrence": n_recurrence,
        "n_no_recurrence": n_no_recurrence,
        "label_col": label_col,
        "id_col": id_col,
    }


def relative_path(path: Path) -> str:
    try:
        return str(path.relative_to(repo_root))
    except ValueError:
        return str(path)


def prediction_files_for_row(row: pd.Series) -> dict[str, Path]:
    run_dir = repo_root / str(row["run_dir"])
    model = str(row["model"])
    analysis_dir = run_dir / "analysis" / model
    files = {
        "wsi": analysis_dir / f"all_predictions_{model}.csv",
        "clinical_fusion": analysis_dir / "fusion" / "fusion_predictions.csv",
    }

    if not files["wsi"].exists():
        matches = sorted(run_dir.glob(f"analysis/**/all_predictions_{model}.csv"))
        if matches:
            files["wsi"] = matches[0]
    if not files["clinical_fusion"].exists():
        matches = sorted(run_dir.glob(f"analysis/{model}/**/fusion_predictions.csv"))
        if matches:
            files["clinical_fusion"] = matches[0]
    return files


case_count_rows = []
for _, row in df.iterrows():
    for result_set, pred_path in prediction_files_for_row(row).items():
        record = {
            "cohort": row["cohort"],
            "source_csv": row["source_csv"],
            "mode": row["mode"],
            "tool_type": row["tool_type"],
            "experiment_base": row["experiment_base"],
            "run_name": row["run_name"],
            "model": row["model"],
            "result_set": result_set,
            "prediction_csv": relative_path(pred_path),
            "prediction_file_found": pred_path.exists(),
        }
        if pred_path.exists():
            record.update(case_counts_from_predictions(pred_path))
        else:
            record.update({
                "n_total": np.nan,
                "n_recurrence": np.nan,
                "n_no_recurrence": np.nan,
                "label_col": np.nan,
                "id_col": np.nan,
            })
        case_count_rows.append(record)

case_counts_df = pd.DataFrame(case_count_rows)
case_counts_df[["n_total", "n_recurrence", "n_no_recurrence"]] = case_counts_df[["n_total", "n_recurrence", "n_no_recurrence"]].astype("Int64")

source_case_counts = (
    case_counts_df
    .dropna(subset=["n_total", "n_recurrence", "n_no_recurrence"])
    .drop_duplicates(["cohort", "source_csv", "result_set", "n_total", "n_recurrence", "n_no_recurrence"])
    .sort_values(["cohort", "source_csv", "result_set"])
)

print("Case counts by benchmark source and result set")
display(source_case_counts[["cohort", "source_csv", "result_set", "n_total", "n_recurrence", "n_no_recurrence"]])

print("Case counts by run/model result row")
display(
    case_counts_df[
        [
            "cohort", "mode", "tool_type", "experiment_base", "run_name", "model",
            "result_set", "n_total", "n_recurrence", "n_no_recurrence",
            "prediction_file_found", "prediction_csv",
        ]
    ].sort_values(["cohort", "mode", "tool_type", "run_name", "model", "result_set"])
)

missing_prediction_files = case_counts_df.loc[~case_counts_df["prediction_file_found"]]
if len(missing_prediction_files):
    print(f"Missing prediction files: {len(missing_prediction_files)}")
    display(missing_prediction_files[["cohort", "run_name", "model", "result_set", "prediction_csv"]])


## Coverage

In [ ]:
coverage = []
for col in group_cols:
    coverage.append(
        df.groupby(["cohort", col], dropna=False)
        .size()
        .rename("n")
        .reset_index()
        .assign(grouping=col)
        .rename(columns={col: "level"})
    )

coverage_df = pd.concat(coverage, ignore_index=True)[["cohort", "grouping", "level", "n"]]
display(coverage_df.sort_values(["cohort", "grouping", "level"]))

In [ ]:
id_cols = ["cohort", "mode", "tool_family", "tool_type", "tile_aggregator", "slide_encoder", "experiment_base", "run_name", "model", "run_dir"]
long_df = df.melt(
    id_vars=[c for c in id_cols if c in df.columns],
    value_vars=metric_cols,
    var_name="metric",
    value_name="auc",
).dropna(subset=["auc"])

long_df["method"] = long_df["metric"].str.replace("_roc_auc", "", regex=False).str.replace("_pr_auc", "", regex=False)
long_df["auc_type"] = np.where(long_df["metric"].str.endswith("roc_auc"), "ROC AUC", "PR AUC")
long_df.head()

## Metric Distributions By Grouping

In [ ]:
def omnibus_pvalue(plot_df: pd.DataFrame, group_col: str, metric: str, order: list[str]) -> tuple[str, float | None]:
    groups = [plot_df.loc[plot_df[group_col] == level, metric].dropna().to_numpy(dtype=float) for level in order]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        return "p=n/a", None
    all_values = np.concatenate(groups)
    if len(np.unique(all_values)) < 2:
        return "Kruskal p=n/a (constant)", None

    try:
        from scipy import stats
    except ImportError:
        return "p=n/a; scipy unavailable", None

    try:
        stat, p = stats.kruskal(*groups, nan_policy="omit")
    except ValueError as exc:
        return f"Kruskal p=n/a ({exc})", None
    if not np.isfinite(p):
        return "Kruskal p=n/a", None

    group_sizes = ", ".join(str(len(g)) for g in groups)
    return f"Kruskal p={p:.3g}; n=[{group_sizes}]", float(p)


def metric_ylim(values: pd.Series, *, top_extra: float = 0.18) -> tuple[float, float]:
    vals = values.dropna().to_numpy()
    if len(vals) == 0:
        return 0.0, 1.0
    lo = float(np.nanmin(vals))
    hi = float(np.nanmax(vals))
    spread = max(hi - lo, 0.02)
    lower = max(0.0, lo - 0.12 * spread)
    upper = min(1.0, hi + top_extra * spread)
    if upper - lower < 0.05:
        center = (upper + lower) / 2
        lower = max(0.0, center - 0.025)
        upper = min(1.0, center + 0.025)
    return lower, upper


def plot_metrics_by_group(group_col: str, *, max_label_len: int = 28):
    levels = df[group_col].dropna().astype(str).unique().tolist()
    if len(levels) <= 1:
        print(f"Skipping {group_col}: only one level")
        return

    fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharey=False)
    axes = axes.ravel()

    for ax, metric in zip(axes, metric_cols):
        plot_df = df[["cohort", group_col, metric]].dropna().copy()
        order = (
            plot_df.groupby(group_col)[metric]
            .median()
            .sort_values(ascending=False)
            .index.tolist()
        )

        if sns is not None:
            sns.boxplot(data=plot_df, x=group_col, y=metric, hue="cohort", order=order, ax=ax, fliersize=0)
            sns.stripplot(data=plot_df, x=group_col, y=metric, hue="cohort", order=order, ax=ax, dodge=True, color="#2f4858", size=3, alpha=0.45, legend=False)
            handles, labels_ = ax.get_legend_handles_labels()
            if handles:
                ax.legend(handles[: plot_df["cohort"].nunique()], labels_[: plot_df["cohort"].nunique()], title="cohort", fontsize=8, title_fontsize=8)
        else:
            values = [plot_df.loc[plot_df[group_col] == x, metric].to_numpy() for x in order]
            ax.boxplot(values, labels=order, showfliers=False)
            for i, vals in enumerate(values, start=1):
                jitter = np.random.default_rng(42).normal(i, 0.035, size=len(vals))
                ax.scatter(jitter, vals, s=18, alpha=0.65, color="#2f4858")

        p_labels = []
        for cohort, cohort_df in plot_df.groupby("cohort"):
            p_label, p_value = omnibus_pvalue(cohort_df, group_col, metric, order)
            p_labels.append(f"{cohort}: {p_label}")
        ax.set_title(f"{metric}\n" + " | ".join(p_labels), fontsize=10)
        ax.set_xlabel("")
        ax.set_ylabel("AUC")
        labels = [str(t.get_text())[:max_label_len] for t in ax.get_xticklabels()]
        ax.set_xticklabels(labels, rotation=35, ha="right")
        lower, upper = metric_ylim(plot_df[metric])
        ax.set_ylim(lower, upper)

    fig.suptitle(f"Metric distributions by {group_col}", y=1.02, fontsize=16)
    fig.tight_layout()
    return fig

for group_col in group_cols:
    plot_metrics_by_group(group_col)
    plt.show()

## Summary Tables

In [ ]:
def summarize_by(group_col: str) -> pd.DataFrame:
    return (
        df.groupby(["cohort", group_col])[metric_cols]
        .agg(["count", "median", "mean", "std", "min", "max"])
        .round(4)
        .sort_index()
    )

summaries = {col: summarize_by(col) for col in group_cols}
for col, table in summaries.items():
    print(f"\n=== {col} ===")
    display(table)

## Median Heatmaps

In [ ]:
def plot_median_heatmap(group_col: str):
    med = df.groupby(["cohort", group_col])[metric_cols].median().sort_index()
    if len(med) <= 1:
        print(f"Skipping {group_col}: only one level")
        return

    fig, ax = plt.subplots(figsize=(10, max(2.5, 0.45 * len(med))))
    if sns is not None:
        sns.heatmap(med, annot=True, fmt=".3f", cmap="viridis", vmin=0.45, vmax=0.8, ax=ax)
    else:
        im = ax.imshow(med.to_numpy(), aspect="auto", vmin=0.45, vmax=0.8, cmap="viridis")
        ax.set_xticks(np.arange(len(med.columns)), med.columns, rotation=35, ha="right")
        ax.set_yticks(np.arange(len(med.index)), med.index)
        fig.colorbar(im, ax=ax, label="Median AUC")
        for i in range(med.shape[0]):
            for j in range(med.shape[1]):
                ax.text(j, i, f"{med.iloc[i, j]:.3f}", ha="center", va="center", color="white")
    ax.set_title(f"Median metric values by {group_col}")
    ax.set_xlabel("")
    ax.set_ylabel("")
    fig.tight_layout()
    return fig

for group_col in group_cols:
    plot_median_heatmap(group_col)
    plt.show()

## Cross-Cohort Agreement

These views compare shared tool types across cohorts. The rank-difference table is useful for spotting approaches that perform consistently well versus approaches that appear cohort-specific.

In [ ]:
def cohort_tool_summary(group_col: str = "tool_type", agg: str = "median") -> pd.DataFrame:
    grouped = df.groupby(["cohort", group_col])[metric_cols]
    if agg == "mean":
        out = grouped.mean()
    elif agg == "median":
        out = grouped.median()
    else:
        raise ValueError("agg must be 'median' or 'mean'")
    return out.reset_index()

tool_summary = cohort_tool_summary("tool_type", agg="median")
display(tool_summary.sort_values(["tool_type", "cohort"]))

In [ ]:
def rank_agreement_table(group_col: str = "tool_type") -> pd.DataFrame:
    rows = []
    summary = cohort_tool_summary(group_col, agg="median")
    for metric in metric_cols:
        wide = summary.pivot(index=group_col, columns="cohort", values=metric).dropna()
        if wide.empty:
            continue
        ranks = wide.rank(ascending=False, method="min")
        cohorts = list(wide.columns)
        if len(cohorts) >= 2:
            rank_delta = ranks[cohorts[0]] - ranks[cohorts[1]]
            value_delta = wide[cohorts[0]] - wide[cohorts[1]]
        else:
            rank_delta = pd.Series(np.nan, index=wide.index)
            value_delta = pd.Series(np.nan, index=wide.index)
        out = wide.copy()
        out.columns = [f"{c}_{metric}" for c in out.columns]
        for cohort in cohorts:
            out[f"{cohort}_{metric}_rank"] = ranks[cohort]
        out["metric"] = metric
        out["abs_rank_delta"] = rank_delta.abs()
        out["value_delta_first_minus_second"] = value_delta
        rows.append(out.reset_index())
    return pd.concat(rows, ignore_index=True)

rank_agreement = rank_agreement_table("tool_type")
display(rank_agreement.sort_values(["metric", "abs_rank_delta", "tool_type"]))

In [ ]:
def plot_cohort_metric_scatter(group_col: str = "tool_type"):
    summary = cohort_tool_summary(group_col, agg="median")
    cohorts = sorted(summary["cohort"].unique())
    if len(cohorts) != 2:
        print(f"Expected exactly two cohorts for scatter comparison; found {cohorts}")
        return
    c0, c1 = cohorts

    fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharex=False, sharey=False)
    axes = axes.ravel()
    for ax, metric in zip(axes, metric_cols):
        wide = summary.pivot(index=group_col, columns="cohort", values=metric).dropna()
        if wide.empty or c0 not in wide.columns or c1 not in wide.columns:
            ax.set_axis_off()
            continue
        ax.scatter(wide[c0], wide[c1], s=55, color="#2f4858", alpha=0.85)
        lo = float(np.nanmin(wide[[c0, c1]].to_numpy()))
        hi = float(np.nanmax(wide[[c0, c1]].to_numpy()))
        pad = max((hi - lo) * 0.08, 0.01)
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color="#777777", linestyle="--", linewidth=1)
        for label, row in wide.iterrows():
            ax.annotate(str(label), (row[c0], row[c1]), xytext=(4, 3), textcoords="offset points", fontsize=8)
        corr = wide[c0].corr(wide[c1], method="spearman") if len(wide) >= 2 else np.nan
        ax.set_title(f"{metric}\nSpearman r={corr:.2f}" if np.isfinite(corr) else metric)
        ax.set_xlabel(c0)
        ax.set_ylabel(c1)
        ax.set_xlim(lo - pad, hi + pad)
        ax.set_ylim(lo - pad, hi + pad)
    fig.suptitle(f"Cross-cohort median metric agreement by {group_col}", y=1.02, fontsize=16)
    fig.tight_layout()
    return fig

plot_cohort_metric_scatter("tool_type")
plt.show()

## Best Rows By Metric

In [ ]:
best_rows = []
for metric in metric_cols:
    cols = ["cohort", "experiment_base", "mode", "tool_type", "tile_aggregator", "slide_encoder", "model", "run_name", metric]
    top = df[cols].sort_values(metric, ascending=False).head(10).copy()
    top.insert(0, "rank", range(1, len(top) + 1))
    top.insert(0, "metric", metric)
    best_rows.append(top.rename(columns={metric: "value"}))

best_by_metric = pd.concat(best_rows, ignore_index=True)
display(best_by_metric)